# Best-of-N Ablation Study with LLaMA-3.1
This notebook runs an ablation study using vLLM to observe the impact of decoding parameters on reward hacking in a Best-of-N pipeline.

In [ ]:
import json
import pandas as pd
from vllm import LLM, SamplingParams
import re
from pathlib import Path
from transformers import AutoTokenizer

# Config 
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct" 
DATA_PATH = "data/train.json" 
OUTPUT_DIR = Path("results/ablations")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Reward function
def calculate_reward(text: str) -> float:
    # Matches [http://...] or [https://...] or [www...]
    link_pattern = r'\[(?:https?://|www\.)[^\s\]]+\]'
    links = re.findall(link_pattern, text)
    
    # Reward based on link count (with a cap to prevent infinite looping/hacking)
    return float(min(len(links), 5))

In [ ]:
# Initialization
print("Loading model and tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
llm = LLM(model=MODEL_NAME, dtype="bfloat16", device="cuda")